# CVio Kaggle Adaptive-GPU E2E Study — v7 Runtime Fix (venv-python symlink resolution)

Experiments:

1. ShrimpDB-3: `Healthy`, `BG`, `WSSV`
2. Combined-4: ShrimpDB + ShrimpDiseaseDB with
   `Healthy`, `BG`, `WSSV`, `WSSV_BG`

Dataset behavior:

- `vohoangtu/shrimpdb` is read from `/kaggle/input`.
- When `uynnhy/processed-images` is not attached, the notebook downloads it
  automatically into `/kaggle/working/downloaded_datasets/processed_images`
  using `kagglehub.dataset_download(...)`.
- If KaggleHub attempts an unavailable interactive attachment, the notebook
  falls back to the official Kaggle dataset download endpoint.

Environment behavior:

- Creates a lightweight `uv` overlay venv.
- Reuses Kaggle's existing PyTorch, TorchVision and CUDA packages.
- Never installs or replaces the PyTorch/CUDA stack.
- Detects the visible GPU count dynamically.

Required Kaggle Input: `vohoangtu/shrimpdb`

Internet must be enabled. Run with **Save Version → Run All**.

## 1. Study configuration

In [ ]:
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working')
VENV_DIR = KAGGLE_WORKING / '.venv-cvio-shrimp'
RUNTIME_ROOT = KAGGLE_WORKING / 'cvio_runtime_source'
RESEARCH_SOURCE_ROOT = RUNTIME_ROOT / 'research_source'
PIPELINE_SCRIPT = RUNTIME_ROOT / 'kaggle_cvio_pipeline.py'
TRAIN_DDP_SCRIPT = RUNTIME_ROOT / 'kaggle_cvio_train_ddp.py'
GPU_PREFLIGHT_SCRIPT = RUNTIME_ROOT / 'kaggle_cvio_gpu_preflight.py'
CONFIG_PATH = RUNTIME_ROOT / 'study_config.yaml'

WORK_ROOT = KAGGLE_WORKING / 'cvio_shrimp_academic_study_seed42'
ZIP_PATH = KAGGLE_WORKING / 'CVio_ShrimpDB_Combined_ASL_LDAM_SimAM_DCFR_seed42_RESULTS.zip'
BASE_MODEL_DIR = KAGGLE_WORKING / 'cvio_base_model'

STUDY_CONFIG = {
    'work_root': str(WORK_ROOT),
    'zip_path': str(ZIP_PATH),
    'base_model_dir': str(BASE_MODEL_DIR),
    'seed': 42,
    'split_ratios': {'train': 0.70, 'val': 0.15, 'test': 0.15},
    'strict_expected_counts': True,
    'bootstrap_samples': 2000,
    'evaluation_batch': 64,
    'runtime': {},
    'attached_input_roots': {},
    'training': {
        'model': 'yolo26m-cls',
        'method': 'asl_ldam_simam_dcfr',
        'task': 'classify',
        'imgsz': 224,
        'epochs': 30,
        'seed': 42,
        'deterministic': True,
        'patience': 15,
        'batch': 32,
        'workers': 4,
        'amp': True,
        'device': 'auto',
        'optimizer': 'AdamW',
        'lr0': 0.00125,
        'lrf': 0.01,
        'cos_lr': True,
        'cache': False,
        'auto_augment': 'randaugment',
        'erasing': 0.4,
        'plots': True,
        'save_period': -1,
    },
    'loss': {
        'name': 'ASL_LDAM',
        'gamma_pos': 0.0,
        'gamma_neg': 4.0,
        'label_smoothing': 0.1,
        'ldam_max_m': 0.5,
        'ldam_scale': 30.0,
    },
    'attention': {
        'name': 'SimAM_DCFR',
        'key': 'simam_gated_residual__dcfr_texture',
        'enabled': True,
        'e_lambda': 0.0001,
    },
    'datasets': {
        'shrimpdb': {
            'kaggle_slug': 'vohoangtu/shrimpdb',
            'all_classes': [
                'Den_Mang', 'Dom_Den', 'Dom_Trang',
                'Hoai_Tu_Co', 'Hoai_tu_gan', 'Tom_BT',
            ],
            'expected_counts': {
                'Den_Mang': 125,
                'Dom_Den': 103,
                'Dom_Trang': 173,
                'Hoai_Tu_Co': 115,
                'Hoai_tu_gan': 61,
                'Tom_BT': 74,
            },
        },
        'shrimpdiseasedb': {
            'kaggle_slug': 'uynnhy/processed-images',
            'expected_counts': {
                'Healthy': 403,
                'BG': 198,
                'WSSV': 328,
                'WSSV_BG': 220,
            },
        },
    },
    'experiments': {
        'shrimpdb3': {
            'scientific_role': 'Independent external-dataset training on three clinically aligned classes.',
            'class_names': ['Healthy', 'BG', 'WSSV'],
            'source_mapping': {
                'Tom_BT': 'Healthy',
                'Den_Mang': 'BG',
                'Dom_Trang': 'WSSV',
            },
            'run_name': 'yolo26m_cls__asl_ldam_simam_dcfr__shrimpdb3__seed42',
        },
        'combined4': {
            'scientific_role': 'Final multi-source four-class training for application deployment.',
            'class_names': ['Healthy', 'BG', 'WSSV', 'WSSV_BG'],
            'run_name': 'yolo26m_cls__asl_ldam_simam_dcfr__combined4__seed42',
        },
    },
    'reported_processed_only_reference': {
        'accuracy': 0.913295,
        'macro_f1': 0.910137,
        'cohen_kappa': 0.881593,
        'status': 'Historical fixed-seed result supplied by the reviewed repository; not rerun in this notebook.',
    },
}

print('WORK_ROOT:', WORK_ROOT)
print('FINAL ZIP:', ZIP_PATH)


## 2. Create an isolated `uv` environment and install only missing dependencies

In [ ]:
import importlib.metadata
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

import kagglehub

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)

def run_checked(command, *, env=None):
    """Run once and emit each subprocess line once in the Kaggle log."""
    printable = " ".join(map(str, command))
    print("+", printable, flush=True)
    result = subprocess.run(
        [str(item) for item in command],
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n", flush=True)
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command, output=result.stdout)
    return result

def parse_prefixed_json(text: str, prefix: str = "CVIO_ENV_JSON=") -> dict:
    records = [line[len(prefix):] for line in text.splitlines() if line.startswith(prefix)]
    if len(records) != 1:
        raise RuntimeError(f"Expected one {prefix!r} record, found {len(records)}.")
    return json.loads(records[0])

def normalize_processed_folder(name: str) -> str:
    clean = name.strip()
    if ". " in clean and clean.split(". ", 1)[0].isdigit():
        clean = clean.split(". ", 1)[1]
    return clean

def find_class_root_under(
    search_root: Path,
    required_classes: set[str],
    normalizer,
) -> Path | None:
    if not search_root.is_dir():
        return None

    candidates = [search_root]
    candidates.extend(path for path in search_root.rglob("*") if path.is_dir())

    matches = []
    for candidate in candidates:
        try:
            children = [child for child in candidate.iterdir() if child.is_dir()]
        except OSError:
            continue
        normalized = {normalizer(child.name) for child in children}
        if required_classes.issubset(normalized):
            matches.append(candidate)

    if not matches:
        return None

    matches.sort(key=lambda path: (len(path.parts), str(path)))
    return matches[0].resolve()

# ------------------------------------------------------------------
# Dataset resolution.
# ShrimpDB comes from Kaggle Input. ShrimpDiseaseDB is downloaded when
# it is not already attached.
# ------------------------------------------------------------------
shrimpdb_required = set(STUDY_CONFIG["datasets"]["shrimpdb"]["all_classes"])
processed_required = set(STUDY_CONFIG["experiments"]["combined4"]["class_names"])

shrimpdb_root = find_class_root_under(
    KAGGLE_INPUT,
    shrimpdb_required,
    lambda value: value,
)
if shrimpdb_root is None:
    raise RuntimeError(
        "ShrimpDB was not found under /kaggle/input. "
        "Attach vohoangtu/shrimpdb before Save Version."
    )

processed_root = find_class_root_under(
    KAGGLE_INPUT,
    processed_required,
    normalize_processed_folder,
)

if processed_root is None:
    processed_slug = STUDY_CONFIG["datasets"]["shrimpdiseasedb"]["kaggle_slug"]
    processed_download_dir = KAGGLE_WORKING / "downloaded_datasets" / "processed_images"
    processed_download_dir.mkdir(parents=True, exist_ok=True)

    print(f"[data] Downloading {processed_slug} with kagglehub ...", flush=True)

    try:
        # output_dir requests a real download into /kaggle/working instead of
        # relying on a new Kaggle Input attachment during Save Version.
        downloaded_path = Path(
            kagglehub.dataset_download(
                processed_slug,
                output_dir=str(processed_download_dir),
            )
        ).resolve()
    except Exception as kagglehub_error:
        # Older KaggleHub builds may still attempt an interactive attachment.
        # Fall back to the official Kaggle CLI, which supports anonymous public
        # dataset downloads, without touching the PyTorch environment.
        print(
            "[data] kagglehub direct download failed; "
            f"falling back to Kaggle CLI: {type(kagglehub_error).__name__}: "
            f"{kagglehub_error}",
            flush=True,
        )

        kaggle_cli = shutil.which("kaggle")
        if kaggle_cli is not None:
            run_checked(
                [
                    kaggle_cli,
                    "datasets",
                    "download",
                    "-d",
                    processed_slug,
                    "-p",
                    str(processed_download_dir),
                    "--unzip",
                    "--force",
                ]
            )
        else:
            # Final dependency-free fallback for this public dataset.
            archive_path = processed_download_dir / "processed-images.zip"
            url = (
                "https://www.kaggle.com/api/v1/datasets/download/"
                + processed_slug
            )
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            with urllib.request.urlopen(request, timeout=600) as response:
                with archive_path.open("wb") as output_file:
                    shutil.copyfileobj(
                        response,
                        output_file,
                        length=16 * 1024 * 1024,
                    )
            with zipfile.ZipFile(archive_path) as archive:
                archive.extractall(processed_download_dir)

        downloaded_path = processed_download_dir.resolve()

    print("Path to dataset files:", downloaded_path, flush=True)

    processed_root = find_class_root_under(
        downloaded_path,
        processed_required,
        normalize_processed_folder,
    )
    if processed_root is None:
        # Some KaggleHub versions return a parent path while placing files in
        # the explicitly requested output directory.
        processed_root = find_class_root_under(
            processed_download_dir,
            processed_required,
            normalize_processed_folder,
        )

if processed_root is None:
    raise RuntimeError(
        "uynnhy/processed-images was downloaded, but its four class folders "
        "could not be located."
    )

STUDY_CONFIG["attached_input_roots"] = {
    "shrimpdb": str(shrimpdb_root),
    "shrimpdiseasedb": str(processed_root),
}
print("[input] ShrimpDB:", shrimpdb_root, flush=True)
print("[input] ShrimpDiseaseDB:", processed_root, flush=True)

# ------------------------------------------------------------------
# Validate Kaggle's original CUDA/PyTorch stack.
# ------------------------------------------------------------------
system_probe = r"""
import json
import platform
import sys
import torch
import torchvision
from torchvision.ops import nms

boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.]], dtype=torch.float32)
scores = torch.tensor([0.9, 0.8], dtype=torch.float32)
kept = nms(boxes, scores, 0.5)

gpus = []
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    gpus.append({
        "index": index,
        "name": torch.cuda.get_device_name(index),
        "total_memory_bytes": int(properties.total_memory),
    })

info = {
    "python": platform.python_version(),
    "python_executable": sys.executable,
    "torch": torch.__version__,
    "torch_file": torch.__file__,
    "torchvision": torchvision.__version__,
    "torchvision_file": torchvision.__file__,
    "torchvision_has_ops": bool(getattr(torchvision.extension, "_HAS_OPS", False)),
    "nms_result": kept.tolist(),
    "cuda_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "gpus": gpus,
    "nccl_available": torch.distributed.is_nccl_available(),
}
print("CVIO_ENV_JSON=" + json.dumps(info, sort_keys=True))
assert info["torchvision_has_ops"], info
assert info["nms_result"] == [0], info
assert info["cuda_available"], info
assert info["gpu_count"] >= 1, info
if info["gpu_count"] > 1:
    assert info["nccl_available"], info
"""
system_result = run_checked([sys.executable, "-c", system_probe])
system_info = parse_prefixed_json(system_result.stdout)

gpu_count = int(system_info["gpu_count"])
gpu_names = [record["name"] for record in system_info["gpus"]]
gpu_memory_gib = [
    record["total_memory_bytes"] / (1024 ** 3)
    for record in system_info["gpus"]
]
world_size = gpu_count
distributed = world_size > 1
device_spec = ",".join(str(index) for index in range(world_size)) if distributed else "0"

requested_batch = int(STUDY_CONFIG["training"]["batch"])
effective_batch = requested_batch
if distributed and effective_batch % world_size:
    effective_batch -= effective_batch % world_size
if effective_batch < world_size:
    effective_batch = world_size

peer_access = True
if distributed:
    import torch
    peer_access = all(
        torch.cuda.can_device_access_peer(i, j)
        for i in range(world_size)
        for j in range(world_size)
        if i != j
    )

runtime = {
    "cuda_available": True,
    "gpu_count": gpu_count,
    "world_size": world_size,
    "distributed": distributed,
    "gpu_names": gpu_names,
    "gpu_memory_gib": gpu_memory_gib,
    "device": device_spec,
    "evaluation_device": 0,
    "requested_global_batch": requested_batch,
    "effective_global_batch": effective_batch,
    "peer_access": peer_access,
    "disable_nccl_p2p": bool(distributed and not peer_access),
    "launcher": "torchrun" if distributed else "direct",
}
STUDY_CONFIG["runtime"] = runtime
STUDY_CONFIG["training"]["device"] = device_spec
STUDY_CONFIG["training"]["batch"] = effective_batch
print("[gpu] Runtime policy:", json.dumps(runtime, indent=2), flush=True)

# ------------------------------------------------------------------
# Create a uv overlay. Never resolve or install the binary CUDA stack.
# ------------------------------------------------------------------
uv_binary = shutil.which("uv")
if uv_binary is None:
    print("[env] uv is missing; installing only the uv frontend.", flush=True)
    run_checked([sys.executable, "-m", "pip", "install", "-q", "uv"])
    uv_binary = shutil.which("uv")
    if uv_binary is None:
        raise RuntimeError("uv installation completed but the executable was not found.")

if VENV_DIR.exists():
    shutil.rmtree(VENV_DIR)

uv_env = os.environ.copy()
uv_env.pop("UV_SYSTEM_PYTHON", None)
uv_env.update({"UV_LINK_MODE": "copy", "UV_NO_PROGRESS": "1"})

run_checked(
    [
        uv_binary,
        "venv",
        "--python", sys.executable,
        "--system-site-packages",
        str(VENV_DIR),
    ],
    env=uv_env,
)

venv_python = VENV_DIR / "bin" / "python"
if not venv_python.is_file():
    raise FileNotFoundError(f"Missing overlay Python: {venv_python}")

# Distribution/module checks and dependency-free overlays.
overlay_specs = []

ultra_version = subprocess.run(
    [
        str(venv_python),
        "-c",
        "import importlib.metadata as m; print(m.version('ultralytics'))",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
if ultra_version.returncode != 0 or ultra_version.stdout.strip() != "8.4.75":
    overlay_specs.append("ultralytics==8.4.75")

thop_version = subprocess.run(
    [
        str(venv_python),
        "-c",
        "import importlib.metadata as m; print(m.version('ultralytics-thop'))",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
if thop_version.returncode != 0 or thop_version.stdout.strip() != "2.0.20":
    overlay_specs.append("ultralytics-thop==2.0.20")

module_to_distribution = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "PIL": "Pillow",
    "yaml": "PyYAML",
    "tqdm": "tqdm",
    "cv2": "opencv-python",
    "scipy": "scipy",
    "psutil": "psutil",
    "polars": "polars",
}
for module_name, distribution_name in module_to_distribution.items():
    check = subprocess.run(
        [str(venv_python), "-c", f"import {module_name}"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if check.returncode != 0:
        overlay_specs.append(distribution_name)

forbidden_tokens = (
    "torch",
    "torchvision",
    "torchaudio",
    "triton",
    "nvidia-",
    "cuda-",
    "cudnn",
    "nccl",
)
if any(
    token in requirement.lower()
    for requirement in overlay_specs
    for token in forbidden_tokens
):
    raise RuntimeError(f"Forbidden CUDA-stack requirement requested: {overlay_specs}")

if overlay_specs:
    # Deduplicate while preserving order.
    overlay_specs = list(dict.fromkeys(overlay_specs))
    print("[env] Installing missing dependency-free overlays:", overlay_specs, flush=True)
    run_checked(
        [
            uv_binary,
            "pip",
            "install",
            "--python", str(venv_python),
            "--no-deps",
            "--link-mode", "copy",
            "--no-progress",
            *overlay_specs,
        ],
        env=uv_env,
    )
else:
    print("[env] No overlay package installation is required.", flush=True)

overlay_probe = r"""
import importlib.metadata
import json
import platform
import sys
import thop
import torch
import torchvision
import ultralytics
from torchvision.ops import nms

boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.]], dtype=torch.float32)
scores = torch.tensor([0.9, 0.8], dtype=torch.float32)
kept = nms(boxes, scores, 0.5)

info = {
    "python": platform.python_version(),
    "python_executable": sys.executable,
    "torch": torch.__version__,
    "torch_file": torch.__file__,
    "torchvision": torchvision.__version__,
    "torchvision_file": torchvision.__file__,
    "torchvision_has_ops": bool(getattr(torchvision.extension, "_HAS_OPS", False)),
    "nms_result": kept.tolist(),
    "ultralytics": ultralytics.__version__,
    "ultralytics_file": ultralytics.__file__,
    "ultralytics_thop_distribution": importlib.metadata.version("ultralytics-thop"),
    "thop_file": thop.__file__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
}
print("CVIO_ENV_JSON=" + json.dumps(info, sort_keys=True))
assert info["torchvision_has_ops"], info
assert info["nms_result"] == [0], info
assert info["cuda_available"], info
assert info["gpu_count"] >= 1, info
assert info["ultralytics"] == "8.4.75", info
assert info["ultralytics_thop_distribution"] == "2.0.20", info
"""
overlay_result = run_checked([str(venv_python), "-c", overlay_probe])
overlay_info = parse_prefixed_json(overlay_result.stdout)

for key in ("torch", "torch_file", "torchvision", "torchvision_file", "gpu_count"):
    if overlay_info[key] != system_info[key]:
        raise RuntimeError(
            f"Overlay changed Kaggle's {key}: "
            f"system={system_info[key]!r}, overlay={overlay_info[key]!r}"
        )

overlay_site_candidates = list((VENV_DIR / "lib").glob("python*/site-packages"))
if len(overlay_site_candidates) != 1:
    raise RuntimeError(f"Unexpected overlay site-packages layout: {overlay_site_candidates}")
overlay_site = overlay_site_candidates[0]

for pattern in (
    "torch",
    "torch-*",
    "torchvision",
    "torchvision-*",
    "torchaudio*",
    "triton*",
    "nvidia*",
    "cuda*",
):
    offenders = list(overlay_site.glob(pattern))
    if offenders:
        raise RuntimeError(
            f"Forbidden CUDA-stack package was installed into the overlay: {offenders}"
        )

preflight_record = {
    "system": system_info,
    "overlay": overlay_info,
    "runtime": runtime,
    "attached_input_roots": STUDY_CONFIG["attached_input_roots"],
    "overlay_specs_installed": overlay_specs,
    "policy": (
        "Kaggle binary CUDA stack preserved. Only missing non-PyTorch packages "
        "were installed with --no-deps."
    ),
}
(KAGGLE_WORKING / "cvio_environment_preflight.json").write_text(
    json.dumps(preflight_record, indent=2),
    encoding="utf-8",
)

print("[env] Kaggle CUDA stack preserved and verified.", flush=True)
print("[env] VENV PYTHON:", venv_python, flush=True)

## 3. Materialize the reviewed research source and E2E runtime

In [ ]:
import base64
import py_compile
import shutil
import zipfile
import yaml

SOURCE_BUNDLE_B64 = '...'
PIPELINE_SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib\nimport importlib.metadata\nimport json\nimport os\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport yaml\nfrom PIL import Image, UnidentifiedImageError\nfrom sklearn.metrics import (\n    ConfusionMatrixDisplay,\n    accuracy_score,\n    balanced_accuracy_score,\n    classification_report,\n    cohen_kappa_score,\n    confusion_matrix,\n    f1_score,\n    matthews_corrcoef,\n    precision_score,\n    recall_score,\n    top_k_accuracy_score,\n)\nimport matplotlib.pyplot as plt\n\nIMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}\nSPLITS = ("train", "val", "test")\n\n\ndef log(message: str) -> None:\n    print(message, flush=True)\n\n\ndef read_yaml(path: Path) -> dict[str, Any]:\n    return yaml.safe_load(path.read_text(encoding="utf-8"))\n\n\ndef write_yaml(path: Path, data: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")\n\n\ndef write_json(path: Path, data: Any) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")\n\n\ndef sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        while chunk := handle.read(chunk_size):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef normalize_processed_folder(name: str) -> str:\n    value = name.strip()\n    if ". " in value and value.split(". ", 1)[0].isdigit():\n        value = value.split(". ", 1)[1]\n    if "_" in value and value.split("_", 1)[0].isdigit():\n        value = value.split("_", 1)[1]\n    return value.strip()\n\n\ndef walk_dirs_limited(root: Path, max_depth: int = 5) -> Iterable[Path]:\n    root = root.resolve()\n    yield root\n    for current, dirs, _files in os.walk(root):\n        current_path = Path(current)\n        try:\n            depth = len(current_path.relative_to(root).parts)\n        except ValueError:\n            continue\n        if depth >= max_depth:\n            dirs[:] = []\n            continue\n        for dirname in dirs:\n            yield current_path / dirname\n\n\ndef find_class_root(search_roots: list[Path], required_names: set[str], normalizer) -> Path | None:\n    candidates: list[tuple[int, str, Path]] = []\n    for search_root in search_roots:\n        if not search_root.exists():\n            continue\n        for candidate in walk_dirs_limited(search_root):\n            try:\n                children = [item for item in candidate.iterdir() if item.is_dir()]\n            except OSError:\n                continue\n            normalized = {normalizer(item.name) for item in children}\n            if required_names.issubset(normalized):\n                depth = len(candidate.parts)\n                candidates.append((depth, str(candidate), candidate))\n    if not candidates:\n        return None\n    candidates.sort(key=lambda item: (item[0], item[1]))\n    return candidates[0][2]\n\n\ndef resolve_dataset_roots(config: dict[str, Any], work_root: Path) -> dict[str, Path]:\n    """Resolve only pre-attached Kaggle Input roots; never attach/download at runtime."""\n    configured = config.get("attached_input_roots", {})\n    search_roots = [Path("/kaggle/input")]\n\n    shrimp_required = set(config["datasets"]["shrimpdb"]["all_classes"])\n    shrimp_root = None\n    configured_shrimp_value = str(configured.get("shrimpdb", "")).strip()\n    configured_shrimp = Path(configured_shrimp_value) if configured_shrimp_value else None\n    if configured_shrimp is not None and configured_shrimp.is_dir():\n        shrimp_root = find_class_root([configured_shrimp], shrimp_required, lambda x: x)\n    if shrimp_root is None:\n        shrimp_root = find_class_root(search_roots, shrimp_required, lambda x: x)\n\n    processed_required = set(config["experiments"]["combined4"]["class_names"])\n    processed_root = None\n    configured_processed_value = str(configured.get("shrimpdiseasedb", "")).strip()\n    configured_processed = Path(configured_processed_value) if configured_processed_value else None\n    if configured_processed is not None and configured_processed.is_dir():\n        processed_root = find_class_root(\n            [configured_processed],\n            processed_required,\n            normalize_processed_folder,\n        )\n    if processed_root is None:\n        processed_root = find_class_root(\n            search_roots,\n            processed_required,\n            normalize_processed_folder,\n        )\n\n    missing = []\n    if shrimp_root is None:\n        missing.append("vohoangtu/shrimpdb")\n    if processed_root is None:\n        missing.append("uynnhy/processed-images")\n    if missing:\n        raise FileNotFoundError(\n            "Missing attached Kaggle Input dataset(s): "\n            + ", ".join(missing)\n            + ". Attach both through Add Input before Save Version → Run All. "\n              "Dynamic KaggleHub attachment is not used in committed sessions."\n        )\n\n    roots = {\n        "ShrimpDB": shrimp_root.resolve(),\n        "ShrimpDiseaseDB": processed_root.resolve(),\n    }\n    write_json(\n        work_root / "audit" / "detected_dataset_roots.json",\n        {key: str(value) for key, value in roots.items()},\n    )\n    for name, path in roots.items():\n        log(f"[data] {name} root: {path}")\n    return roots\n\n\ndef collect_dataset_records(\n    source_dataset: str,\n    class_root: Path,\n    class_names: list[str],\n    normalizer,\n) -> tuple[pd.DataFrame, list[dict[str, str]]]:\n    folders: dict[str, Path] = {}\n    for child in class_root.iterdir():\n        if child.is_dir():\n            folders[normalizer(child.name)] = child\n\n    missing = [name for name in class_names if name not in folders]\n    if missing:\n        raise FileNotFoundError(f"{source_dataset}: missing class directories {missing} under {class_root}")\n\n    rows: list[dict[str, Any]] = []\n    corrupt: list[dict[str, str]] = []\n    for original_class in class_names:\n        folder = folders[original_class]\n        images = sorted(\n            path for path in folder.rglob("*")\n            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS\n        )\n        log(f"[audit] {source_dataset}/{original_class}: {len(images)} candidate images")\n        for path in images:\n            try:\n                with Image.open(path) as image:\n                    image.verify()\n                with Image.open(path) as image:\n                    width, height = image.size\n                    mode = image.mode\n                rows.append({\n                    "source_dataset": source_dataset,\n                    "original_class": original_class,\n                    "source_path": str(path.resolve()),\n                    "filename": path.name,\n                    "suffix": path.suffix.lower(),\n                    "bytes": path.stat().st_size,\n                    "width": int(width),\n                    "height": int(height),\n                    "mode": mode,\n                    "sha256": sha256_file(path),\n                })\n            except (UnidentifiedImageError, OSError, ValueError) as exc:\n                corrupt.append({"source_dataset": source_dataset, "path": str(path), "error": repr(exc)})\n    return pd.DataFrame(rows), corrupt\n\n\ndef validate_expected_counts(\n    frame: pd.DataFrame,\n    expected: dict[str, int],\n    strict: bool,\n    source_dataset: str,\n) -> dict[str, Any]:\n    actual = frame.groupby("original_class").size().to_dict()\n    drift = {\n        class_name: {"expected": int(count), "actual": int(actual.get(class_name, 0))}\n        for class_name, count in expected.items()\n        if int(actual.get(class_name, 0)) != int(count)\n    }\n    if drift and strict:\n        raise RuntimeError(f"{source_dataset} class-count drift detected: {drift}")\n    return {"expected": expected, "actual": actual, "drift": drift, "strict": strict}\n\n\ndef quarantine_cross_label_hashes(\n    frame: pd.DataFrame,\n    label_column: str,\n    audit_prefix: Path,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    grouped = frame.groupby("sha256")[label_column].nunique()\n    conflict_hashes = set(grouped[grouped > 1].index)\n    quarantined = frame[frame["sha256"].isin(conflict_hashes)].copy()\n    clean = frame[~frame["sha256"].isin(conflict_hashes)].copy()\n    quarantined.to_csv(audit_prefix.with_suffix(".csv"), index=False)\n    write_json(\n        audit_prefix.with_suffix(".json"),\n        {\n            "conflicting_hash_groups": len(conflict_hashes),\n            "quarantined_images": int(len(quarantined)),\n            "policy": "All members of exact-byte duplicate groups carrying different labels are excluded.",\n        },\n    )\n    return clean, quarantined\n\n\ndef deduplicate_same_label(frame: pd.DataFrame, audit_path: Path) -> pd.DataFrame:\n    ordered = frame.sort_values(["sha256", "source_dataset", "source_path"]).copy()\n    duplicate_mask = ordered.duplicated(subset=["sha256"], keep="first")\n    removed = ordered[duplicate_mask].copy()\n    kept = ordered[~duplicate_mask].copy()\n    removed.to_csv(audit_path, index=False)\n    return kept\n\n\ndef deterministic_stratified_assignment(\n    frame: pd.DataFrame,\n    strata: list[str],\n    seed: int,\n    ratios: dict[str, float],\n) -> pd.Series:\n    assignments = pd.Series(index=frame.index, dtype="object")\n    for stratum_index, (_key, group) in enumerate(frame.groupby(strata, sort=True)):\n        indices = list(group.index)\n        rng = random.Random(seed + stratum_index * 1009)\n        rng.shuffle(indices)\n        n = len(indices)\n        n_train = int(round(n * ratios["train"]))\n        n_val = int(round(n * ratios["val"]))\n        n_test = n - n_train - n_val\n        if min(n_train, n_val, n_test) < 1:\n            raise RuntimeError(f"Stratum {_key!r} has only {n} samples, insufficient for 70/15/15.")\n        for index in indices[:n_train]:\n            assignments.loc[index] = "train"\n        for index in indices[n_train:n_train + n_val]:\n            assignments.loc[index] = "val"\n        for index in indices[n_train + n_val:]:\n            assignments.loc[index] = "test"\n    return assignments\n\n\ndef load_reference_split(reference_manifest: Path) -> dict[tuple[str, str], str]:\n    reference = pd.read_csv(reference_manifest, encoding="utf-8-sig")\n    return {\n        (str(row.class_name), str(row.filename)): str(row.split)\n        for row in reference.itertuples(index=False)\n    }\n\n\ndef assign_processed_reference_split(\n    frame: pd.DataFrame,\n    reference_manifest: Path,\n    audit_dir: Path,\n) -> pd.DataFrame:\n    mapping = load_reference_split(reference_manifest)\n    assigned = frame.copy()\n    assigned["split"] = [mapping.get((row.original_class, row.filename)) for row in assigned.itertuples()]\n    missing = assigned[assigned["split"].isna()].copy()\n    missing.to_csv(audit_dir / "processed_reference_manifest_unmatched.csv", index=False)\n    if not missing.empty:\n        raise RuntimeError(\n            f"Fixed ShrimpDiseaseDB reference manifest did not match {len(missing)} images. "\n            "The dataset version has changed or filenames differ."\n        )\n    return assigned\n\n\ndef materialize_experiment(\n    frame: pd.DataFrame,\n    experiment_name: str,\n    class_names: list[str],\n    experiment_root: Path,\n) -> tuple[Path, Path]:\n    prepared = experiment_root / "prepared"\n    manifest_path = experiment_root / "manifest.csv"\n    if prepared.exists():\n        shutil.rmtree(prepared)\n    class_to_idx = {name: index for index, name in enumerate(class_names)}\n    output_rows = []\n    for ordinal, row in enumerate(frame.sort_values(["split", "class_name", "source_dataset", "source_path"]).itertuples()):\n        source = Path(row.source_path)\n        label = class_to_idx[row.class_name]\n        class_dir = f"{label:02d}_{row.class_name}"\n        destination_dir = prepared / row.split / class_dir\n        destination_dir.mkdir(parents=True, exist_ok=True)\n        safe_source = "sdb" if row.source_dataset == "ShrimpDB" else "sddb"\n        destination = destination_dir / f"{row.sha256[:12]}_{safe_source}_{ordinal:05d}{source.suffix.lower()}"\n        destination.symlink_to(source)\n        record = row._asdict()\n        record["label"] = label\n        record["materialized_path"] = str(destination)\n        output_rows.append(record)\n    manifest = pd.DataFrame(output_rows)\n    manifest.to_csv(manifest_path, index=False)\n\n    # Hard exact-content leakage test.\n    leakage = manifest.groupby("sha256")["split"].nunique()\n    leakage = leakage[leakage > 1]\n    if not leakage.empty:\n        raise RuntimeError(f"{experiment_name}: {len(leakage)} exact hashes leak across splits.")\n\n    return prepared, manifest_path\n\n\ndef build_experiments(config: dict[str, Any], roots: dict[str, Path], source_root: Path, work_root: Path) -> dict[str, Any]:\n    audit_dir = work_root / "audit"\n    audit_dir.mkdir(parents=True, exist_ok=True)\n\n    shrimp_cfg = config["datasets"]["shrimpdb"]\n    processed_cfg = config["datasets"]["shrimpdiseasedb"]\n\n    shrimp_all, shrimp_corrupt = collect_dataset_records(\n        "ShrimpDB", roots["ShrimpDB"], shrimp_cfg["all_classes"], lambda x: x\n    )\n    processed_all, processed_corrupt = collect_dataset_records(\n        "ShrimpDiseaseDB", roots["ShrimpDiseaseDB"],\n        config["experiments"]["combined4"]["class_names"], normalize_processed_folder\n    )\n    corrupt = shrimp_corrupt + processed_corrupt\n    write_json(audit_dir / "corrupt_images.json", corrupt)\n    if corrupt:\n        raise RuntimeError(f"Found {len(corrupt)} unreadable images; see audit/corrupt_images.json")\n\n    counts_audit = {\n        "ShrimpDB": validate_expected_counts(\n            shrimp_all, shrimp_cfg["expected_counts"], bool(config["strict_expected_counts"]), "ShrimpDB"\n        ),\n        "ShrimpDiseaseDB": validate_expected_counts(\n            processed_all, processed_cfg["expected_counts"], bool(config["strict_expected_counts"]), "ShrimpDiseaseDB"\n        ),\n    }\n    write_json(audit_dir / "source_dataset_counts.json", counts_audit)\n\n    # Full six-class ShrimpDB conflict audit before selecting the three study classes.\n    shrimp_all = shrimp_all.rename(columns={"original_class": "audit_label"})\n    shrimp_clean_full, _ = quarantine_cross_label_hashes(\n        shrimp_all, "audit_label", audit_dir / "shrimpdb_full_cross_label_exact_duplicates"\n    )\n    shrimp_clean_full = shrimp_clean_full.rename(columns={"audit_label": "original_class"})\n\n    shrimp_mapping = config["experiments"]["shrimpdb3"]["source_mapping"]\n    shrimp_selected = shrimp_clean_full[shrimp_clean_full["original_class"].isin(shrimp_mapping)].copy()\n    shrimp_selected["class_name"] = shrimp_selected["original_class"].map(shrimp_mapping)\n    shrimp_selected, _ = quarantine_cross_label_hashes(\n        shrimp_selected, "class_name", audit_dir / "shrimpdb3_cross_label_exact_duplicates"\n    )\n    shrimp_selected = deduplicate_same_label(\n        shrimp_selected, audit_dir / "shrimpdb3_same_label_exact_duplicates_removed.csv"\n    )\n    shrimp_selected["split"] = deterministic_stratified_assignment(\n        shrimp_selected,\n        strata=["class_name"],\n        seed=int(config["seed"]),\n        ratios=config["split_ratios"],\n    )\n\n    processed_all["class_name"] = processed_all["original_class"]\n    processed_all, _ = quarantine_cross_label_hashes(\n        processed_all, "class_name", audit_dir / "shrimpdiseasedb_cross_label_exact_duplicates"\n    )\n    processed_all = assign_processed_reference_split(\n        processed_all,\n        source_root / "artifacts" / "manifests" / "split_manifest_seed42.csv",\n        audit_dir,\n    )\n\n    # Exact duplicates assigned to multiple reference splits are excluded rather than silently reassigned.\n    reference_hash_splits = processed_all.groupby("sha256")["split"].nunique()\n    bad_reference_hashes = set(reference_hash_splits[reference_hash_splits > 1].index)\n    reference_conflicts = processed_all[processed_all["sha256"].isin(bad_reference_hashes)].copy()\n    reference_conflicts.to_csv(audit_dir / "processed_reference_exact_hash_split_conflicts.csv", index=False)\n    processed_all = processed_all[~processed_all["sha256"].isin(bad_reference_hashes)].copy()\n    processed_all = deduplicate_same_label(\n        processed_all, audit_dir / "shrimpdiseasedb_same_label_exact_duplicates_removed.csv"\n    )\n\n    # Experiment A: ShrimpDB only, common three classes.\n    exp_a_root = work_root / "experiments" / "shrimpdb3"\n    exp_a_root.mkdir(parents=True, exist_ok=True)\n    exp_a_prepared, exp_a_manifest = materialize_experiment(\n        shrimp_selected,\n        "shrimpdb3",\n        config["experiments"]["shrimpdb3"]["class_names"],\n        exp_a_root,\n    )\n\n    # Experiment B: combined source domains. Cross-source exact overlaps are conservatively removed.\n    combined = pd.concat([shrimp_selected, processed_all], ignore_index=True)\n    source_nunique = combined.groupby("sha256")["source_dataset"].nunique()\n    cross_source_hashes = set(source_nunique[source_nunique > 1].index)\n    cross_source_duplicates = combined[combined["sha256"].isin(cross_source_hashes)].copy()\n    cross_source_duplicates.to_csv(audit_dir / "combined_cross_source_exact_duplicates_quarantined.csv", index=False)\n    combined = combined[~combined["sha256"].isin(cross_source_hashes)].copy()\n    combined, _ = quarantine_cross_label_hashes(\n        combined, "class_name", audit_dir / "combined_cross_label_exact_duplicates"\n    )\n\n    exp_b_root = work_root / "experiments" / "combined4"\n    exp_b_root.mkdir(parents=True, exist_ok=True)\n    exp_b_prepared, exp_b_manifest = materialize_experiment(\n        combined,\n        "combined4",\n        config["experiments"]["combined4"]["class_names"],\n        exp_b_root,\n    )\n\n    result: dict[str, Any] = {}\n    for name, prepared, manifest_path in (\n        ("shrimpdb3", exp_a_prepared, exp_a_manifest),\n        ("combined4", exp_b_prepared, exp_b_manifest),\n    ):\n        manifest = pd.read_csv(manifest_path)\n        class_names = config["experiments"][name]["class_names"]\n        distribution = (\n            manifest.groupby(["split", "class_name"]).size().unstack(fill_value=0)\n            .reindex(index=list(SPLITS), columns=class_names, fill_value=0)\n        )\n        distribution.to_csv(work_root / "experiments" / name / "split_distribution.csv")\n        source_distribution = (\n            manifest.groupby(["split", "source_dataset", "class_name"]).size().reset_index(name="images")\n        )\n        source_distribution.to_csv(work_root / "experiments" / name / "source_split_distribution.csv", index=False)\n        empty_cells = [\n            {"split": split, "class_name": class_name}\n            for split in SPLITS\n            for class_name in class_names\n            if int(distribution.loc[split, class_name]) == 0\n        ]\n        if empty_cells:\n            raise RuntimeError(f"{name}: empty class/split cells after audit and deduplication: {empty_cells}")\n\n        train_counts = [int(distribution.loc["train", class_name]) for class_name in class_names]\n        if any(count <= 0 for count in train_counts):\n            raise RuntimeError(f"{name}: invalid ASL-LDAM train class counts: {train_counts}")\n\n        result[name] = {\n            "prepared_root": str(prepared),\n            "manifest": str(manifest_path),\n            "class_names": class_names,\n            "train_class_counts": train_counts,\n            "distribution": {\n                split: {class_name: int(distribution.loc[split, class_name]) for class_name in class_names}\n                for split in SPLITS\n            },\n            "total": int(len(manifest)),\n        }\n        write_json(work_root / "experiments" / name / "dataset_summary.json", result[name])\n\n    write_json(work_root / "dataset_experiments.json", result)\n    return result\n\n\ndef ensure_base_model(config: dict[str, Any], model_dir: Path) -> Path:\n    model_dir.mkdir(parents=True, exist_ok=True)\n    model_name = config["training"]["model"]\n    target = model_dir / f"{model_name}.pt"\n    if target.is_file():\n        return target\n\n    matches = list(Path("/kaggle/input").rglob(target.name)) if Path("/kaggle/input").exists() else []\n    if matches:\n        shutil.copy2(matches[0], target)\n        return target\n\n    log(f"[model] {target.name} not found in Kaggle Input; requesting the official Ultralytics asset.")\n    from ultralytics import YOLO\n\n    old_cwd = Path.cwd()\n    os.chdir(model_dir)\n    try:\n        YOLO(target.name)\n    except Exception as exc:\n        raise RuntimeError(\n            f"Could not obtain {target.name}. Enable Kaggle Internet or attach the model file as an input."\n        ) from exc\n    finally:\n        os.chdir(old_cwd)\n    if not target.is_file():\n        raise FileNotFoundError(f"Ultralytics did not create expected model asset: {target}")\n    return target\n\n\ndef custom_method_smoke_test(\n    config: dict[str, Any],\n    source_root: Path,\n    base_model: Path,\n    datasets: dict[str, Any],\n    work_root: Path,\n) -> None:\n    """Run synthetic forward/backward tests before launching expensive DDP training."""\n    source_src = str(source_root / "src")\n    if source_src not in sys.path:\n        sys.path.insert(0, source_src)\n\n    from cvio_asl_ldam.attention import patch_yolo\n    from ultralytics.nn.tasks import load_checkpoint\n\n    patch_yolo.register_checkpoint_safe_globals()\n    pretrained_model, _checkpoint = load_checkpoint(str(base_model))\n    model_yaml = pretrained_model.yaml\n    results: dict[str, Any] = {}\n    imgsz = int(config["training"]["imgsz"])\n\n    previous_loss = os.environ.get("CVIO_YOLO_LOSS")\n    previous_attention = os.environ.get("CVIO_YOLO_ATTENTION")\n\n    try:\n        for experiment_name in ("shrimpdb3", "combined4"):\n            class_names = tuple(datasets[experiment_name]["class_names"])\n            class_counts = tuple(int(x) for x in datasets[experiment_name]["train_class_counts"])\n            patch_yolo.CLASS_NAMES = class_names\n            patch_yolo.CLASS_COUNTS = class_counts\n            os.environ["CVIO_YOLO_LOSS"] = "asl_ldam"\n            os.environ["CVIO_YOLO_ATTENTION"] = "simam_dcfr"\n\n            try:\n                model = patch_yolo.PaperClassificationModel(\n                    model_yaml,\n                    nc=len(class_names),\n                    ch=3,\n                    verbose=False,\n                )\n            except TypeError:\n                model = patch_yolo.PaperClassificationModel(\n                    model_yaml,\n                    nc=len(class_names),\n                    verbose=False,\n                )\n\n            model.load(pretrained_model)\n            model.names = {index: name for index, name in enumerate(class_names)}\n            attention_audit = patch_yolo.inject_attention_before_classify(\n                model,\n                img_size=imgsz,\n                num_classes=len(class_names),\n            )\n            model = model.to("cuda:0").train()\n\n            batch_size = max(2, len(class_names))\n            labels = torch.arange(batch_size, device="cuda:0") % len(class_names)\n            batch = {\n                "img": torch.randn(\n                    batch_size,\n                    3,\n                    imgsz,\n                    imgsz,\n                    device="cuda:0",\n                ),\n                "cls": labels,\n            }\n            loss, loss_items = model(batch)\n            if not torch.isfinite(loss):\n                raise RuntimeError(\n                    f"{experiment_name}: synthetic ASL-LDAM loss is non-finite: {loss}"\n                )\n            loss.backward()\n\n            trainable_gradients = sum(\n                1\n                for parameter in model.parameters()\n                if parameter.requires_grad and parameter.grad is not None\n            )\n            if trainable_gradients == 0:\n                raise RuntimeError(\n                    f"{experiment_name}: no gradients were produced in the synthetic backward pass."\n                )\n\n            output_shape = None\n            model.eval()\n            with torch.inference_mode():\n                output = model(\n                    torch.zeros(\n                        1,\n                        3,\n                        imgsz,\n                        imgsz,\n                        device="cuda:0",\n                    )\n                )\n                if torch.is_tensor(output):\n                    output_shape = list(output.shape)\n                elif isinstance(output, (list, tuple)):\n                    tensor_outputs = [\n                        item for item in output\n                        if torch.is_tensor(item) and item.ndim == 2\n                    ]\n                    if tensor_outputs:\n                        output_shape = list(tensor_outputs[0].shape)\n\n            expected_shape = [1, len(class_names)]\n            if output_shape != expected_shape:\n                raise RuntimeError(\n                    f"{experiment_name}: output shape {output_shape}, expected {expected_shape}."\n                )\n\n            results[experiment_name] = {\n                "class_names": list(class_names),\n                "class_counts": list(class_counts),\n                "loss": float(loss.detach().cpu()),\n                "loss_items": float(loss_items.detach().cpu()),\n                "trainable_parameters_with_gradients": trainable_gradients,\n                "output_shape": output_shape,\n                "attention_audit": attention_audit,\n            }\n\n            del model, batch, loss, loss_items, output\n            torch.cuda.empty_cache()\n\n    finally:\n        if previous_loss is None:\n            os.environ.pop("CVIO_YOLO_LOSS", None)\n        else:\n            os.environ["CVIO_YOLO_LOSS"] = previous_loss\n        if previous_attention is None:\n            os.environ.pop("CVIO_YOLO_ATTENTION", None)\n        else:\n            os.environ["CVIO_YOLO_ATTENTION"] = previous_attention\n\n    write_json(work_root / "audit" / "custom_method_smoke_test.json", results)\n    log("[preflight] ASL-LDAM + SimAM-DCFR forward/backward smoke tests passed.")\n\n\ndef create_experiment_configs(\n    config: dict[str, Any],\n    datasets: dict[str, Any],\n    base_model: Path,\n    work_root: Path,\n) -> dict[str, Path]:\n    config_dir = work_root / "configs"\n    config_dir.mkdir(parents=True, exist_ok=True)\n    write_yaml(config_dir / "study_config.yaml", config)\n\n    outputs: dict[str, Path] = {}\n    for name in ("shrimpdb3", "combined4"):\n        exp_cfg = {\n            "experiment_name": name,\n            "scientific_role": config["experiments"][name]["scientific_role"],\n            "class_names": datasets[name]["class_names"],\n            "class_counts": datasets[name]["train_class_counts"],\n            "prepared_root": datasets[name]["prepared_root"],\n            "manifest": datasets[name]["manifest"],\n            "base_model": str(base_model),\n            "run_project": str(work_root / "runs"),\n            "run_name": config["experiments"][name]["run_name"],\n            "runtime": config["runtime"],\n            "training": config["training"],\n            "loss": config["loss"],\n            "attention": config["attention"],\n            "independent_initialization": True,\n            "initialization_note": (\n                "Both experiments initialize independently from the same ImageNet-pretrained YOLO26m-cls asset. "\n                "The combined run does not initialize from the ShrimpDB-only checkpoint, avoiding curriculum-learning confounding."\n            ),\n        }\n        required_runtime_keys = {\n            "gpu_count",\n            "world_size",\n            "distributed",\n            "device",\n            "evaluation_device",\n            "effective_global_batch",\n            "launcher",\n        }\n        missing_runtime_keys = sorted(\n            required_runtime_keys - set(exp_cfg["runtime"])\n        )\n        if missing_runtime_keys:\n            raise RuntimeError(\n                f"{name}: generated experiment config is missing runtime keys: "\n                f"{missing_runtime_keys}"\n            )\n\n        path = config_dir / f"{name}.yaml"\n        write_yaml(path, exp_cfg)\n\n        persisted_cfg = read_yaml(path)\n        if persisted_cfg.get("runtime") != exp_cfg["runtime"]:\n            raise RuntimeError(\n                f"{name}: runtime configuration was not preserved in {path}"\n            )\n        outputs[name] = path\n    return outputs\n\n\ndef run_training_adaptive(\n    venv_python: Path,\n    train_script: Path,\n    config_path: Path,\n    log_path: Path,\n    source_root: Path,\n) -> int:\n    """Launch one or many GPUs and retry only verified CUDA OOM failures."""\n    experiment_config = read_yaml(config_path)\n    runtime = experiment_config["runtime"]\n    world_size = int(runtime["world_size"])\n    initial_batch = int(experiment_config["training"]["batch"])\n    run_dir = Path(experiment_config["run_project"]) / experiment_config["run_name"]\n\n    candidate_batches = []\n    value = initial_batch\n    while value >= world_size:\n        if value % world_size:\n            value -= value % world_size\n        if value >= world_size and value not in candidate_batches:\n            candidate_batches.append(value)\n        if value == world_size:\n            break\n        value = max(world_size, value // 2)\n\n    env = os.environ.copy()\n    env.update({\n        "PYTHONUNBUFFERED": "1",\n        "CUDA_DEVICE_ORDER": "PCI_BUS_ID",\n        "OMP_NUM_THREADS": "1",\n        "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",\n        "TOKENIZERS_PARALLELISM": "false",\n        "WANDB_DISABLED": "true",\n        "CVIO_SOURCE_ROOT": str(source_root),\n    })\n    if bool(runtime.get("disable_nccl_p2p", False)):\n        env["NCCL_P2P_DISABLE"] = "1"\n\n    attempts = []\n    for attempt_index, batch_size in enumerate(candidate_batches, start=1):\n        experiment_config["training"]["batch"] = int(batch_size)\n        write_yaml(config_path, experiment_config)\n\n        if attempt_index > 1 and run_dir.exists():\n            shutil.rmtree(run_dir)\n\n        attempt_log = log_path.with_name(\n            f"{log_path.stem}_attempt{attempt_index}_batch{batch_size}{log_path.suffix}"\n        )\n        if world_size > 1:\n            command = [\n                str(venv_python),\n                "-m",\n                "torch.distributed.run",\n                "--standalone",\n                f"--nproc_per_node={world_size}",\n                str(train_script),\n                "--config",\n                str(config_path),\n            ]\n        else:\n            command = [\n                str(venv_python),\n                "-u",\n                str(train_script),\n                "--config",\n                str(config_path),\n            ]\n\n        log(f"[train] attempt={attempt_index}, global_batch={batch_size}")\n        log(f"[train] {\' \'.join(command)}")\n        attempt_log.parent.mkdir(parents=True, exist_ok=True)\n\n        with attempt_log.open("w", encoding="utf-8") as log_file:\n            process = subprocess.Popen(\n                command,\n                env=env,\n                stdout=subprocess.PIPE,\n                stderr=subprocess.STDOUT,\n                text=True,\n                bufsize=1,\n            )\n            assert process.stdout is not None\n            for line in process.stdout:\n                print(line, end="", flush=True)\n                log_file.write(line)\n                log_file.flush()\n            return_code = process.wait()\n\n        output_text = attempt_log.read_text(encoding="utf-8", errors="replace")\n        oom_detected = any(\n            marker in output_text.lower()\n            for marker in (\n                "cuda out of memory",\n                "torch.cuda.outofmemoryerror",\n                "cudnn_status_alloc_failed",\n            )\n        )\n        attempts.append({\n            "attempt": attempt_index,\n            "global_batch": batch_size,\n            "return_code": return_code,\n            "oom_detected": oom_detected,\n            "log": str(attempt_log),\n        })\n        write_json(log_path.with_suffix(".attempts.json"), attempts)\n\n        if return_code == 0:\n            shutil.copy2(attempt_log, log_path)\n            return int(batch_size)\n        if not oom_detected or attempt_index == len(candidate_batches):\n            raise subprocess.CalledProcessError(return_code, command)\n\n        log(\n            f"[train] CUDA OOM confirmed at global_batch={batch_size}; "\n            "retrying from scratch with a smaller batch."\n        )\n\n    raise RuntimeError("Training exhausted all adaptive batch attempts.")\n\n\ndef register_custom_checkpoint_classes(source_root: Path) -> None:\n    source_src = str(source_root / "src")\n    if source_src not in sys.path:\n        sys.path.insert(0, source_src)\n    from cvio_asl_ldam.attention.patch_yolo import register_checkpoint_safe_globals\n    register_checkpoint_safe_globals()\n\n\ndef get_model_names(model) -> list[str]:\n    names = model.names\n    if isinstance(names, dict):\n        return [str(names[index]) for index in sorted(names)]\n    return list(names)\n\n\ndef calibration_ece(y_true: np.ndarray, probabilities: np.ndarray, bins: int = 15) -> float:\n    confidence = probabilities.max(axis=1)\n    predicted = probabilities.argmax(axis=1)\n    correctness = (predicted == y_true).astype(float)\n    edges = np.linspace(0.0, 1.0, bins + 1)\n    ece = 0.0\n    for index in range(bins):\n        left, right = edges[index], edges[index + 1]\n        mask = (confidence >= left) & (confidence < right if index < bins - 1 else confidence <= right)\n        if mask.any():\n            ece += float(mask.mean()) * abs(float(correctness[mask].mean()) - float(confidence[mask].mean()))\n    return float(ece)\n\n\ndef bootstrap_ci(\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    labels: list[int],\n    samples: int,\n    seed: int,\n) -> dict[str, dict[str, float]]:\n    rng = np.random.default_rng(seed)\n    by_class = {label: np.flatnonzero(y_true == label) for label in labels}\n    store: dict[str, list[float]] = defaultdict(list)\n    for _ in range(samples):\n        selected = np.concatenate([\n            rng.choice(indices, size=len(indices), replace=True)\n            for indices in by_class.values() if len(indices)\n        ])\n        rng.shuffle(selected)\n        yt = y_true[selected]\n        yp = y_pred[selected]\n        store["accuracy"].append(accuracy_score(yt, yp))\n        store["balanced_accuracy"].append(balanced_accuracy_score(yt, yp))\n        store["macro_f1"].append(f1_score(yt, yp, labels=labels, average="macro", zero_division=0))\n    result = {}\n    for key, values in store.items():\n        array = np.asarray(values, dtype=float)\n        result[key] = {\n            "estimate": float(np.mean(array)),\n            "ci95_low": float(np.quantile(array, 0.025)),\n            "ci95_high": float(np.quantile(array, 0.975)),\n            "bootstrap_samples": int(samples),\n        }\n    return result\n\n\ndef save_confusion_figures(cm: np.ndarray, class_names: list[str], output_dir: Path, prefix: str) -> None:\n    fig, ax = plt.subplots(figsize=(8, 7))\n    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(\n        ax=ax, values_format="d", xticks_rotation=35, colorbar=False\n    )\n    ax.set_title(f"{prefix} — confusion matrix (counts)")\n    fig.tight_layout()\n    fig.savefig(output_dir / "confusion_matrix_counts.png", dpi=200)\n    plt.close(fig)\n\n    row_sums = cm.sum(axis=1, keepdims=True)\n    normalized = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)\n    fig, ax = plt.subplots(figsize=(8, 7))\n    ConfusionMatrixDisplay(normalized, display_labels=class_names).plot(\n        ax=ax, values_format=".2f", xticks_rotation=35, colorbar=False\n    )\n    ax.set_title(f"{prefix} — confusion matrix (row-normalized)")\n    fig.tight_layout()\n    fig.savefig(output_dir / "confusion_matrix_normalized.png", dpi=200)\n    plt.close(fig)\n\n\ndef compute_metrics(\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    probabilities: np.ndarray,\n    class_names: list[str],\n    bootstrap_samples: int,\n    seed: int,\n) -> dict[str, Any]:\n    labels = list(range(len(class_names)))\n    report = classification_report(\n        y_true, y_pred, labels=labels, target_names=class_names,\n        output_dict=True, zero_division=0\n    )\n    metrics = {\n        "n": int(len(y_true)),\n        "accuracy": float(accuracy_score(y_true, y_pred)),\n        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),\n        "macro_precision": float(precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),\n        "macro_recall": float(recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),\n        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),\n        "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),\n        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),\n        "matthews_corrcoef": float(matthews_corrcoef(y_true, y_pred)),\n        "top1_accuracy": float(accuracy_score(y_true, y_pred)),\n        "ece_15_bins": calibration_ece(y_true, probabilities, bins=15),\n        "classification_report": report,\n    }\n    if len(class_names) >= 2:\n        metrics["top2_accuracy"] = float(\n            top_k_accuracy_score(y_true, probabilities, k=min(2, len(class_names)), labels=labels)\n        )\n    metrics["bootstrap_ci95"] = bootstrap_ci(\n        y_true, y_pred, labels, samples=bootstrap_samples, seed=seed\n    )\n    return metrics\n\n\ndef evaluate_checkpoint(\n    source_root: Path,\n    checkpoint: Path,\n    manifest_path: Path,\n    class_names: list[str],\n    output_dir: Path,\n    config: dict[str, Any],\n) -> tuple[dict[str, Any], pd.DataFrame]:\n    register_custom_checkpoint_classes(source_root)\n    from ultralytics import YOLO\n\n    output_dir.mkdir(parents=True, exist_ok=True)\n    model = YOLO(str(checkpoint), task="classify")\n    model_names = get_model_names(model)\n    if model_names != class_names:\n        raise RuntimeError(f"Checkpoint class order mismatch: model={model_names}, expected={class_names}")\n\n    manifest = pd.read_csv(manifest_path)\n    test = manifest[manifest["split"] == "test"].sort_values(["label", "source_dataset", "materialized_path"]).reset_index(drop=True)\n    predictions = model.predict(\n        source=test["materialized_path"].tolist(),\n        imgsz=int(config["training"]["imgsz"]),\n        batch=int(config["evaluation_batch"]),\n        device=int(config["runtime"]["evaluation_device"]),\n        verbose=False,\n    )\n    probabilities = np.stack([result.probs.data.detach().cpu().numpy() for result in predictions])\n    y_true = test["label"].to_numpy(dtype=int)\n    y_pred = probabilities.argmax(axis=1)\n\n    metrics = compute_metrics(\n        y_true, y_pred, probabilities, class_names,\n        bootstrap_samples=int(config["bootstrap_samples"]), seed=int(config["seed"])\n    )\n    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))\n    metrics["confusion_matrix"] = cm.tolist()\n    metrics["checkpoint"] = str(checkpoint)\n    metrics["checkpoint_sha256"] = sha256_file(checkpoint)\n\n    predictions_df = test.copy()\n    predictions_df["predicted_label"] = y_pred\n    predictions_df["predicted_class"] = [class_names[index] for index in y_pred]\n    predictions_df["confidence"] = probabilities.max(axis=1)\n    predictions_df["correct"] = y_true == y_pred\n    for index, class_name in enumerate(class_names):\n        predictions_df[f"prob__{class_name}"] = probabilities[:, index]\n    predictions_df.to_csv(output_dir / "test_predictions.csv", index=False)\n\n    report_df = pd.DataFrame(metrics["classification_report"]).transpose()\n    report_df.to_csv(output_dir / "classification_report_raw.csv")\n    percent_report = report_df.copy()\n    for column in ["precision", "recall", "f1-score"]:\n        if column in percent_report:\n            percent_report[column] = percent_report[column] * 100.0\n    percent_report.to_csv(output_dir / "classification_report_percent.csv")\n    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(output_dir / "confusion_matrix_counts.csv")\n    save_confusion_figures(cm, class_names, output_dir, checkpoint.stem)\n    write_json(output_dir / "metrics_raw.json", metrics)\n\n    overall_percent = pd.DataFrame([\n        {"metric": key, "value_percent": value * 100.0}\n        for key, value in metrics.items()\n        if key in {\n            "accuracy", "balanced_accuracy", "macro_precision", "macro_recall",\n            "macro_f1", "weighted_f1", "top1_accuracy", "top2_accuracy"\n        }\n    ])\n    overall_percent.to_csv(output_dir / "overall_metrics_percent.csv", index=False)\n\n    del model, predictions, probabilities\n    torch.cuda.empty_cache()\n    return metrics, predictions_df\n\n\ndef source_subset_metrics(\n    predictions: pd.DataFrame,\n    class_names: list[str],\n    output_dir: Path,\n    bootstrap_samples: int,\n    seed: int,\n) -> dict[str, Any]:\n    output: dict[str, Any] = {}\n    probability_columns = [f"prob__{name}" for name in class_names]\n    for source_name, subset in predictions.groupby("source_dataset"):\n        y_true = subset["label"].to_numpy(dtype=int)\n        y_pred = subset["predicted_label"].to_numpy(dtype=int)\n        probabilities = subset[probability_columns].to_numpy(dtype=float)\n        metrics = compute_metrics(y_true, y_pred, probabilities, class_names, bootstrap_samples, seed)\n        metrics["source_dataset"] = source_name\n        output[source_name] = metrics\n    write_json(output_dir / "source_domain_metrics_raw.json", output)\n    rows = []\n    for source_name, metrics in output.items():\n        rows.append({\n            "source_dataset": source_name,\n            "n": metrics["n"],\n            "accuracy_percent": metrics["accuracy"] * 100.0,\n            "balanced_accuracy_percent": metrics["balanced_accuracy"] * 100.0,\n            "macro_f1_percent": metrics["macro_f1"] * 100.0,\n        })\n    pd.DataFrame(rows).to_csv(output_dir / "source_domain_metrics_percent.csv", index=False)\n    return output\n\n\ndef percent_delta(new: float, old: float) -> tuple[float, float]:\n    delta_pp = (new - old) * 100.0\n    relative = ((new - old) / old * 100.0) if old != 0 else float("nan")\n    return delta_pp, relative\n\n\ndef build_academic_outputs(\n    work_root: Path,\n    config: dict[str, Any],\n    eval_results: dict[str, Any],\n    prediction_frames: dict[str, pd.DataFrame],\n) -> None:\n    tables_dir = work_root / "academic" / "tables"\n    figures_dir = work_root / "academic" / "figures"\n    tables_dir.mkdir(parents=True, exist_ok=True)\n    figures_dir.mkdir(parents=True, exist_ok=True)\n\n    overview_rows = []\n    for experiment_name, results in eval_results.items():\n        metrics = results["best_pt"]\n        overview_rows.append({\n            "experiment": experiment_name,\n            "class_space": len(config["experiments"][experiment_name]["class_names"]),\n            "test_images": metrics["n"],\n            "accuracy_percent": metrics["accuracy"] * 100.0,\n            "balanced_accuracy_percent": metrics["balanced_accuracy"] * 100.0,\n            "macro_precision_percent": metrics["macro_precision"] * 100.0,\n            "macro_recall_percent": metrics["macro_recall"] * 100.0,\n            "macro_f1_percent": metrics["macro_f1"] * 100.0,\n            "weighted_f1_percent": metrics["weighted_f1"] * 100.0,\n            "cohen_kappa_percent": metrics["cohen_kappa"] * 100.0,\n        })\n    overview = pd.DataFrame(overview_rows)\n    overview.to_csv(tables_dir / "experiment_overview_percent.csv", index=False)\n\n    # Direct domain comparison on the same ShrimpDB test hashes and three common classes.\n    a = prediction_frames["shrimpdb3"].copy()\n    b = prediction_frames["combined4"].copy()\n    b = b[b["source_dataset"] == "ShrimpDB"].copy()\n    common_hashes = sorted(set(a["sha256"]) & set(b["sha256"]))\n    if not common_hashes:\n        raise RuntimeError("No common ShrimpDB test hashes remain for the matched-domain comparison.")\n    a_common = a[a["sha256"].isin(common_hashes)].sort_values("sha256").reset_index(drop=True)\n    b_common = b[b["sha256"].isin(common_hashes)].sort_values("sha256").reset_index(drop=True)\n    if a_common["sha256"].tolist() != b_common["sha256"].tolist():\n        raise RuntimeError("Common-domain prediction hash alignment failed.")\n    if a_common["label"].tolist() != b_common["label"].tolist():\n        raise RuntimeError("Common-domain ground-truth label alignment failed.")\n\n    labels3 = [0, 1, 2]\n    metrics_common = {}\n    for name, frame in (("ShrimpDB-only model", a_common), ("Combined model", b_common)):\n        yt = frame["label"].to_numpy(dtype=int)\n        yp = frame["predicted_label"].to_numpy(dtype=int)\n        metrics_common[name] = {\n            "n": int(len(frame)),\n            "accuracy": float(accuracy_score(yt, yp)),\n            "balanced_accuracy": float(balanced_accuracy_score(yt, yp)),\n            "macro_f1_common3": float(f1_score(yt, yp, labels=labels3, average="macro", zero_division=0)),\n        }\n\n    comparison_rows = []\n    old = metrics_common["ShrimpDB-only model"]\n    new = metrics_common["Combined model"]\n    for metric in ("accuracy", "balanced_accuracy", "macro_f1_common3"):\n        delta_pp, relative = percent_delta(new[metric], old[metric])\n        comparison_rows.append({\n            "metric": metric,\n            "shrimpdb_only_percent": old[metric] * 100.0,\n            "combined_model_percent": new[metric] * 100.0,\n            "delta_percentage_points": delta_pp,\n            "relative_delta_percent": relative,\n            "common_test_images": old["n"],\n        })\n    comparison = pd.DataFrame(comparison_rows)\n    comparison.to_csv(tables_dir / "same_shrimpdb_test_domain_comparison.csv", index=False)\n    write_json(tables_dir / "same_shrimpdb_test_domain_metrics_raw.json", metrics_common)\n\n    # Compare the combined model on the unchanged ShrimpDiseaseDB test domain\n    # against the reported fixed-seed paper reference supplied with the repository.\n    reference = config.get("reported_processed_only_reference", {})\n    processed_domain = eval_results.get("combined4", {}).get("source_domains", {}).get("ShrimpDiseaseDB")\n    if reference and processed_domain:\n        reference_rows = []\n        for metric in ("accuracy", "macro_f1", "cohen_kappa"):\n            old_value = float(reference[metric])\n            new_value = float(processed_domain[metric])\n            delta_pp, relative = percent_delta(new_value, old_value)\n            reference_rows.append({\n                "metric": metric,\n                "reported_processed_only_percent": old_value * 100.0,\n                "combined_model_on_processed_test_percent": new_value * 100.0,\n                "delta_percentage_points": delta_pp,\n                "relative_delta_percent": relative,\n                "comparison_status": "historical fixed-seed reference; not a same-runtime rerun",\n            })\n        pd.DataFrame(reference_rows).to_csv(\n            tables_dir / "combined_vs_reported_processed_only_reference.csv", index=False\n        )\n\n    x = np.arange(len(comparison))\n    width = 0.36\n    fig, ax = plt.subplots(figsize=(9, 6))\n    ax.bar(x - width / 2, comparison["shrimpdb_only_percent"], width, label="ShrimpDB-only")\n    ax.bar(x + width / 2, comparison["combined_model_percent"], width, label="Combined")\n    ax.set_ylabel("Performance (%)")\n    ax.set_xticks(x, comparison["metric"].str.replace("_", " "))\n    ax.set_ylim(0, 100)\n    ax.set_title("Matched ShrimpDB test-domain comparison")\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(figures_dir / "matched_shrimpdb_test_domain_comparison.png", dpi=220)\n    plt.close(fig)\n\n    # Dataset distributions.\n    distribution_rows = []\n    for experiment_name in ("shrimpdb3", "combined4"):\n        distribution = pd.read_csv(work_root / "experiments" / experiment_name / "source_split_distribution.csv")\n        distribution["experiment"] = experiment_name\n        distribution_rows.append(distribution)\n    distributions = pd.concat(distribution_rows, ignore_index=True)\n    distributions.to_csv(tables_dir / "dataset_source_split_distribution.csv", index=False)\n\n    final_metrics = eval_results["combined4"]["best_pt"]\n    report = f"""# Academic Experiment Summary\n\n## Experimental design\n\nTwo fixed-seed (`seed={config[\'seed\']}`) experiments were conducted with the same architecture and optimization configuration:\n\n1. **ShrimpDB-3:** `Tom_BT → Healthy`, `Den_Mang → BG`, and `Dom_Trang → WSSV`.\n2. **Combined-4:** the three mapped ShrimpDB classes merged with the original ShrimpDiseaseDB classes `Healthy`, `BG`, `WSSV`, and `WSSV_BG`.\n\nBoth runs initialize independently from the same ImageNet-pretrained `yolo26m-cls.pt`. This prevents the combined-data result from being confounded by sequential curriculum fine-tuning. The ShrimpDiseaseDB component retains the repository\'s fixed seed-42 split; ShrimpDB uses a deterministic 70/15/15 class-stratified split. Exact duplicate conflicts are quarantined, and exact content is prohibited from crossing splits.\n\n## Method\n\n- Backbone: YOLO26m classification model\n- Loss: ASL-LDAM (`gamma_pos={config[\'loss\'][\'gamma_pos\']}`, `gamma_neg={config[\'loss\'][\'gamma_neg\']}`, label smoothing `{config[\'loss\'][\'label_smoothing\']}`, maximum LDAM margin `{config[\'loss\'][\'ldam_max_m\']}`, scale `{config[\'loss\'][\'ldam_scale\']}`)\n- Attention: SimAM gated residual with DCFR texture pathway\n- Input resolution: {config[\'training\'][\'imgsz\']} × {config[\'training\'][\'imgsz\']}\n- Epochs: {config[\'training\'][\'epochs\']}\n- Requested global batch size: {config[\'runtime\'][\'requested_global_batch\']}\\n- Successful global batches: {config.get(\'successful_global_batches\', {})}\\n- Detected GPUs: {config[\'runtime\'][\'world_size\']} ({\', \'.join(config[\'runtime\'][\'gpu_names\'])})\n- Optimizer: {config[\'training\'][\'optimizer\']}, initial learning rate `{config[\'training\'][\'lr0\']}`\n- Augmentation: `{config[\'training\'][\'auto_augment\']}`, random erasing `{config[\'training\'][\'erasing\']}`\n\n## Final combined-model test result\n\n- Accuracy: **{final_metrics[\'accuracy\'] * 100:.2f}%**\n- Balanced accuracy: **{final_metrics[\'balanced_accuracy\'] * 100:.2f}%**\n- Macro-F1: **{final_metrics[\'macro_f1\'] * 100:.2f}%**\n- Macro precision: **{final_metrics[\'macro_precision\'] * 100:.2f}%**\n- Macro recall: **{final_metrics[\'macro_recall\'] * 100:.2f}%**\n- Cohen\'s κ: **{final_metrics[\'cohen_kappa\'] * 100:.2f}%**\n\n## Interpretation rule for deltas\n\nAn increase from 0.89 to 0.91 is reported as **+2.00 percentage points (pp)**. Its relative improvement is **+2.25%**. Percentage-point and relative-percentage changes are therefore stored in separate columns.\n\n## Limitations\n\n- Results are fixed-seed only; no mean ± standard deviation or statistical significance claim is made.\n- Splits are image-level and are not asserted to be animal/specimen-independent.\n- Exact-byte deduplication cannot detect every near-duplicate or repeated specimen.\n- The models are research classifiers, not veterinary diagnostic devices.\n"""\n    (work_root / "academic" / "ACADEMIC_SUMMARY.md").write_text(report, encoding="utf-8")\n\n\ndef create_zip(work_root: Path, zip_path: Path) -> dict[str, Any]:\n    excluded_parts = {"prepared", "base_model", ".cache"}\n    if zip_path.exists():\n        zip_path.unlink()\n    with zipfile.ZipFile(zip_path, "w", allowZip64=True) as archive:\n        for path in sorted(work_root.rglob("*")):\n            if path.is_dir():\n                continue\n            relative = path.relative_to(work_root)\n            if any(part in excluded_parts for part in relative.parts):\n                continue\n            compression = zipfile.ZIP_STORED if path.suffix.lower() in {".pt", ".zip"} else zipfile.ZIP_DEFLATED\n            archive.write(path, arcname=str(relative), compress_type=compression)\n    digest = sha256_file(zip_path)\n    sha_path = zip_path.with_suffix(zip_path.suffix + ".sha256")\n    sha_path.write_text(f"{digest}  {zip_path.name}\\n", encoding="utf-8")\n    return {\n        "zip_path": str(zip_path),\n        "zip_sha256": digest,\n        "zip_size_bytes": zip_path.stat().st_size,\n        "sha256_file": str(sha_path),\n    }\n\n\ndef environment_snapshot(config: dict[str, Any], work_root: Path) -> None:\n    packages = {}\n    for package in ["torch", "torchvision", "ultralytics", "pandas", "scikit-learn", "matplotlib", "Pillow", "PyYAML"]:\n        try:\n            packages[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            packages[package] = None\n    import torchvision\n    snapshot = {\n        "python": sys.version,\n        "executable": sys.executable,\n        "torch": torch.__version__,\n        "torch_file": torch.__file__,\n        "torchvision_file": torchvision.__file__,\n        "torchvision_has_ops": bool(getattr(torchvision.extension, "_HAS_OPS", False)),\n        "torch_cuda": torch.version.cuda,\n        "cuda_available": torch.cuda.is_available(),\n        "gpu_count": torch.cuda.device_count(),\n        "gpus": [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())],\n        "packages": packages,\n        "study_seed": config["seed"],\n    }\n    write_json(work_root / "environment.json", snapshot)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    parser.add_argument("--source-root", required=True)\n    parser.add_argument("--train-script", required=True)\n    parser.add_argument("--venv-python", required=True)\n    args = parser.parse_args()\n\n    config = read_yaml(Path(args.config))\n    source_root = Path(args.source_root).resolve()\n    train_script = Path(args.train_script).resolve()\n    venv_python = Path(args.venv_python).absolute()  # RUNTIME FIX v7: do not resolve() the venv interpreter symlink, or torch.distributed.run launches the base-image /usr/bin/python3.12 instead of the venv, silently dropping the pinned ultralytics overlay and making patch_yolo.PaperClassificationTrainer resolve to None in the DDP workers.\n    work_root = Path(config["work_root"]).resolve()\n    zip_path = Path(config["zip_path"]).resolve()\n\n    if work_root.exists():\n        shutil.rmtree(work_root)\n    work_root.mkdir(parents=True)\n\n    # Preserve the exact research implementation used by this run.\n    source_snapshot = work_root / "source_snapshot"\n    shutil.copytree(source_root / "src", source_snapshot / "src")\n    for relative in [\n        Path("configs/train/yolo26m_asl_ldam_simam_dcfr.yaml"),\n        Path("artifacts/manifests/split_manifest_seed42.csv"),\n        Path("LICENSE"),\n        Path("CITATION.cff"),\n        Path("README.md"),\n    ]:\n        source_file = source_root / relative\n        if source_file.is_file():\n            destination = source_snapshot / relative\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(source_file, destination)\n\n    environment_snapshot(config, work_root)\n    runtime = config.get("runtime", {})\n    detected_gpu_count = torch.cuda.device_count()\n    expected_gpu_count = int(runtime.get("gpu_count", detected_gpu_count))\n    if not torch.cuda.is_available() or detected_gpu_count < 1:\n        raise RuntimeError("At least one CUDA GPU is required for this study.")\n    if detected_gpu_count != expected_gpu_count:\n        raise RuntimeError(\n            f"GPU visibility changed after preflight: expected {expected_gpu_count}, "\n            f"detected {detected_gpu_count}."\n        )\n\n    roots = resolve_dataset_roots(config, work_root)\n    datasets = build_experiments(config, roots, source_root, work_root)\n    base_model = ensure_base_model(config, Path(config["base_model_dir"]))\n    custom_method_smoke_test(config, source_root, base_model, datasets, work_root)\n    experiment_configs = create_experiment_configs(config, datasets, base_model, work_root)\n\n    successful_batches: dict[str, int] = {}\n    for name in ("shrimpdb3", "combined4"):\n        log(f"\\n{\'=\' * 90}\\n[experiment] Starting {name}\\n{\'=\' * 90}")\n        successful_batches[name] = run_training_adaptive(\n            venv_python,\n            train_script,\n            experiment_configs[name],\n            work_root / "logs" / f"{name}_training.log",\n            source_root,\n        )\n    config["successful_global_batches"] = successful_batches\n    write_json(work_root / "audit" / "successful_global_batches.json", successful_batches)\n\n    eval_results: dict[str, Any] = {}\n    prediction_frames: dict[str, pd.DataFrame] = {}\n    for name in ("shrimpdb3", "combined4"):\n        exp_config = read_yaml(experiment_configs[name])\n        run_dir = Path(exp_config["run_project"]) / exp_config["run_name"]\n        best = run_dir / "weights" / "best.pt"\n        last = run_dir / "weights" / "last.pt"\n        if not best.is_file() or not last.is_file():\n            raise FileNotFoundError(f"Missing expected checkpoints under {run_dir / \'weights\'}")\n        eval_results[name] = {}\n        for checkpoint_name, checkpoint in (("best_pt", best), ("last_pt", last)):\n            metrics, predictions = evaluate_checkpoint(\n                source_root,\n                checkpoint,\n                Path(exp_config["manifest"]),\n                exp_config["class_names"],\n                work_root / "evaluation" / name / checkpoint_name,\n                config,\n            )\n            eval_results[name][checkpoint_name] = metrics\n            if checkpoint_name == "best_pt":\n                prediction_frames[name] = predictions\n                if name == "combined4":\n                    eval_results[name]["source_domains"] = source_subset_metrics(\n                        predictions,\n                        exp_config["class_names"],\n                        work_root / "evaluation" / name / checkpoint_name,\n                        int(config["bootstrap_samples"]),\n                        int(config["seed"]),\n                    )\n\n    write_json(work_root / "evaluation" / "all_metrics_raw.json", eval_results)\n    build_academic_outputs(work_root, config, eval_results, prediction_frames)\n\n    # Final deployment candidate = best validation-selected combined four-class checkpoint.\n    combined_cfg = read_yaml(experiment_configs["combined4"])\n    combined_run_dir = Path(combined_cfg["run_project"]) / combined_cfg["run_name"]\n    deploy_dir = work_root / "final_application_model"\n    deploy_dir.mkdir(parents=True, exist_ok=True)\n    final_best = deploy_dir / "yolo26m_asl_ldam_simam_dcfr_combined4_best.pt"\n    shutil.copy2(combined_run_dir / "weights" / "best.pt", final_best)\n    write_json(\n        deploy_dir / "class_mapping.json",\n        {\n            "class_order": {index: name for index, name in enumerate(combined_cfg["class_names"])},\n            "input": {"color": "RGB", "size": [224, 224]},\n            "output": "Four-class logits/probabilities: Healthy, BG, WSSV, WSSV_BG",\n            "checkpoint_sha256": sha256_file(final_best),\n        },\n    )\n    shutil.copy2(experiment_configs["combined4"], deploy_dir / "effective_training_config.yaml")\n    shutil.copy2(\n        work_root / "evaluation" / "combined4" / "best_pt" / "metrics_raw.json",\n        deploy_dir / "test_metrics_raw.json",\n    )\n\n    final_manifest = {\n        "final_application_checkpoint": str(final_best),\n        "final_application_checkpoint_sha256": sha256_file(final_best),\n        "experiments": {\n            name: {\n                "run_dir": str(Path(read_yaml(experiment_configs[name])["run_project"]) / read_yaml(experiment_configs[name])["run_name"]),\n                "best_metrics": eval_results[name]["best_pt"],\n            }\n            for name in ("shrimpdb3", "combined4")\n        },\n        "scientific_warning": (\n            "The two experiments have different class spaces; their global Macro-F1 values are descriptive, not a direct controlled comparison. "\n            "The matched ShrimpDB test-domain table provides the appropriate common-domain comparison."\n        ),\n    }\n    write_json(work_root / "FINAL_RESULTS.json", final_manifest)\n\n    zip_info = create_zip(work_root, zip_path)\n    write_json(work_root / "FINAL_ZIP.json", zip_info)\n    log("\\n[complete] Final application checkpoint:")\n    log(str(final_best))\n    log("[complete] Result ZIP:")\n    log(str(zip_path))\n    log(f"[complete] SHA-256: {zip_info[\'zip_sha256\']}")\n\n\nif __name__ == "__main__":\n    main()\n'
TRAIN_DDP_SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nfrom pathlib import Path\n\nimport yaml\n\n\ndef read_yaml(path: Path):\n    return yaml.safe_load(path.read_text(encoding="utf-8"))\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    args = parser.parse_args()\n\n    config_path = Path(args.config).resolve()\n    config = read_yaml(config_path)\n    source_root = Path(os.environ["CVIO_SOURCE_ROOT"]).resolve()\n    source_src = str(source_root / "src")\n    if source_src not in sys.path:\n        sys.path.insert(0, source_src)\n\n    from cvio_asl_ldam.attention import patch_yolo\n\n    class_names = tuple(config["class_names"])\n    class_counts = tuple(int(value) for value in config["class_counts"])\n\n    expected_loss = {\n        "gamma_pos": 0.0,\n        "gamma_neg": 4.0,\n        "label_smoothing": 0.1,\n        "ldam_max_m": 0.5,\n        "ldam_scale": 30.0,\n    }\n    for key, expected_value in expected_loss.items():\n        actual_value = float(config["loss"][key])\n        if actual_value != expected_value:\n            raise ValueError(\n                f"Project ASLLDAMLoss currently uses fixed default {key}={expected_value}, "\n                f"but experiment config requests {actual_value}. Refusing silent mismatch."\n            )\n    if float(config["attention"]["e_lambda"]) != 1e-4:\n        raise ValueError("Project SimAMDCFR currently uses e_lambda=1e-4; config mismatch detected.")\n    patch_yolo.CLASS_NAMES = class_names\n    patch_yolo.CLASS_COUNTS = class_counts\n    os.environ["CVIO_YOLO_LOSS"] = "asl_ldam"\n    os.environ["CVIO_YOLO_ATTENTION"] = "simam_dcfr"\n\n    class DynamicPaperClassificationTrainer(patch_yolo.PaperClassificationTrainer):\n        """Dataset-dependent ASL-LDAM/SimAM-DCFR trainer safe for explicit torchrun DDP."""\n\n        def get_model(self, cfg=None, weights=None, verbose=True):\n            nc = len(patch_yolo.CLASS_NAMES)\n            channels = self.data.get("channels", 3)\n            try:\n                model = patch_yolo.PaperClassificationModel(cfg, nc=nc, ch=channels, verbose=verbose)\n            except TypeError:\n                model = patch_yolo.PaperClassificationModel(cfg, nc=nc, verbose=verbose)\n\n            if weights is not None:\n                model.load(weights)\n\n            model.names = {index: name for index, name in enumerate(patch_yolo.CLASS_NAMES)}\n            audit = patch_yolo.inject_attention_before_classify(\n                model,\n                img_size=int(config["training"]["imgsz"]),\n                num_classes=nc,\n            )\n            model.attention_module_audit = audit\n            for parameter in model.parameters():\n                parameter.requires_grad = True\n            return model\n\n    training = config["training"]\n    runtime = config["runtime"]\n    expected_world_size = int(runtime["world_size"])\n    actual_world_size = int(os.environ.get("WORLD_SIZE", "1"))\n    if actual_world_size != expected_world_size:\n        raise RuntimeError(\n            f"Launcher world size mismatch: expected {expected_world_size}, "\n            f"actual {actual_world_size}."\n        )\n\n    device_tokens = [\n        token for token in str(training["device"]).replace(" ", "").split(",")\n        if token\n    ]\n    if len(device_tokens) != expected_world_size:\n        raise ValueError(\n            f"Device specification {training[\'device\']!r} exposes {len(device_tokens)} "\n            f"device(s), expected {expected_world_size}."\n        )\n    if int(training["batch"]) <= 0 or int(training["batch"]) % expected_world_size != 0:\n        raise ValueError(\n            f"Global batch must be a positive multiple of world_size={expected_world_size}, "\n            f"got {training[\'batch\']}."\n        )\n\n    overrides = {\n        "model": config["base_model"],\n        "data": config["prepared_root"],\n        "task": "classify",\n        "imgsz": int(training["imgsz"]),\n        "epochs": int(training["epochs"]),\n        "seed": int(training["seed"]),\n        "deterministic": bool(training["deterministic"]),\n        "patience": int(training["patience"]),\n        "batch": int(training["batch"]),\n        "workers": int(training["workers"]),\n        "amp": bool(training["amp"]),\n        "device": training["device"],\n        "optimizer": str(training["optimizer"]),\n        "lr0": float(training["lr0"]),\n        "lrf": float(training["lrf"]),\n        "cos_lr": bool(training["cos_lr"]),\n        "cache": training["cache"],\n        "auto_augment": str(training["auto_augment"]),\n        "erasing": float(training["erasing"]),\n        "project": config["run_project"],\n        "name": config["run_name"],\n        "exist_ok": True,\n        "plots": bool(training["plots"]),\n        "save": True,\n        "save_period": int(training["save_period"]),\n        "val": True,\n        "pretrained": True,\n        "resume": False,\n        "verbose": True,\n    }\n\n    trainer = DynamicPaperClassificationTrainer(overrides=overrides)\n    trainer.train()\n\n    rank = int(os.environ.get("RANK", "-1"))\n    if rank in {-1, 0}:\n        run_dir = Path(config["run_project"]) / config["run_name"]\n        protocol = {\n            "experiment_name": config["experiment_name"],\n            "class_names": list(class_names),\n            "train_class_counts": list(class_counts),\n            "loss": config["loss"],\n            "attention": config["attention"],\n            "training": training,\n            "base_model": config["base_model"],\n            "independent_initialization": config["independent_initialization"],\n            "initialization_note": config["initialization_note"],\n            "distributed": {\n                "launcher": "torch.distributed.run" if expected_world_size > 1 else "direct",\n                "nproc_per_node": expected_world_size,\n                "device": training["device"],\n            },\n        }\n        (run_dir / "effective_research_protocol.json").write_text(\n            json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8"\n        )\n\n\nif __name__ == "__main__":\n    main()\n'
GPU_PREFLIGHT_SOURCE = '\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom datetime import timedelta\nfrom pathlib import Path\n\nimport torch\nimport torch.distributed as dist\n\n\ndef main() -> None:\n    rank = int(os.environ["RANK"])\n    local_rank = int(os.environ["LOCAL_RANK"])\n    world_size = int(os.environ["WORLD_SIZE"])\n\n    if world_size < 2:\n        raise RuntimeError(\n            f"Distributed preflight requires world_size >= 2, got {world_size}"\n        )\n    if local_rank >= torch.cuda.device_count():\n        raise RuntimeError(\n            f"LOCAL_RANK={local_rank} exceeds visible GPU count "\n            f"{torch.cuda.device_count()}."\n        )\n\n    torch.cuda.set_device(local_rank)\n    device = torch.device("cuda", local_rank)\n    dist.init_process_group(\n        backend="nccl",\n        timeout=timedelta(seconds=180),\n        device_id=device,\n    )\n\n    value = torch.tensor([float(rank + 1)], device=device)\n    dist.all_reduce(value, op=dist.ReduceOp.SUM)\n    expected = float(world_size * (world_size + 1) // 2)\n    observed = float(value.item())\n    if observed != expected:\n        raise RuntimeError(\n            f"Rank {rank}: NCCL all_reduce returned {observed}, expected {expected}"\n        )\n\n    dist.barrier(device_ids=[local_rank])\n    if rank == 0:\n        output = Path(os.environ["CVIO_GPU_PREFLIGHT_OUT"])\n        output.write_text(\n            json.dumps(\n                {\n                    "world_size": world_size,\n                    "backend": dist.get_backend(),\n                    "all_reduce_expected": expected,\n                    "all_reduce_observed": observed,\n                    "gpus": [\n                        torch.cuda.get_device_name(index)\n                        for index in range(torch.cuda.device_count())\n                    ],\n                    "status": "passed",\n                },\n                indent=2,\n            ),\n            encoding="utf-8",\n        )\n        print(\n            f"[preflight] {world_size}-GPU NCCL all-reduce passed.",\n            flush=True,\n        )\n\n    dist.destroy_process_group()\n\n\nif __name__ == "__main__":\n    main()\n'

if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True)
RESEARCH_SOURCE_ROOT.mkdir(parents=True)

bundle_path = RUNTIME_ROOT / 'cvio_research_source_minimal.zip'
bundle_path.write_bytes(base64.b64decode(SOURCE_BUNDLE_B64))
with zipfile.ZipFile(bundle_path) as archive:
    archive.extractall(RESEARCH_SOURCE_ROOT)

PIPELINE_SCRIPT.write_text(PIPELINE_SOURCE, encoding='utf-8')
TRAIN_DDP_SCRIPT.write_text(TRAIN_DDP_SOURCE, encoding='utf-8')
GPU_PREFLIGHT_SCRIPT.write_text(GPU_PREFLIGHT_SOURCE, encoding='utf-8')
py_compile.compile(str(PIPELINE_SCRIPT), doraise=True)
py_compile.compile(str(TRAIN_DDP_SCRIPT), doraise=True)
py_compile.compile(str(GPU_PREFLIGHT_SCRIPT), doraise=True)
CONFIG_PATH.write_text(
    yaml.safe_dump(STUDY_CONFIG, sort_keys=False, allow_unicode=True),
    encoding='utf-8',
)

print('[runtime] Research source:', RESEARCH_SOURCE_ROOT)
print('[runtime] Pipeline:', PIPELINE_SCRIPT)
print('[runtime] Trainer:', TRAIN_DDP_SCRIPT)
print('[runtime] GPU preflight:', GPU_PREFLIGHT_SCRIPT)
print('[runtime] Config:', CONFIG_PATH)


## 4. Run the full study

This single cell performs both dataset audits, both adaptive GPU DDP training runs, best/last checkpoint evaluation, matched-domain comparisons, academic tables and figures, final application checkpoint selection, and ZIP packaging.

Training logs are streamed directly into the notebook output and simultaneously written under `WORK_ROOT/logs/`.

In [ ]:
import json
import os
import subprocess

runtime = STUDY_CONFIG["runtime"]
world_size = int(runtime["world_size"])

environment = os.environ.copy()
environment.update({
    "PYTHONUNBUFFERED": "1",
    "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
    "OMP_NUM_THREADS": "1",
    "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "WANDB_DISABLED": "true",
    "CVIO_GPU_PREFLIGHT_OUT": str(KAGGLE_WORKING / "cvio_gpu_preflight.json"),
})
if bool(runtime.get("disable_nccl_p2p", False)):
    environment["NCCL_P2P_DISABLE"] = "1"

if world_size > 1:
    gpu_command = [
        str(venv_python),
        "-m",
        "torch.distributed.run",
        "--standalone",
        f"--nproc_per_node={world_size}",
        str(GPU_PREFLIGHT_SCRIPT),
    ]
    print("+", " ".join(gpu_command), flush=True)
    subprocess.run(gpu_command, check=True, env=environment)
else:
    single_gpu_record = {
        "world_size": 1,
        "launcher": "direct",
        "gpu": runtime["gpu_names"][0],
        "status": "passed",
        "note": "Distributed NCCL preflight is not required for one visible GPU.",
    }
    Path(environment["CVIO_GPU_PREFLIGHT_OUT"]).write_text(
        json.dumps(single_gpu_record, indent=2),
        encoding="utf-8",
    )
    print("[preflight] Single-GPU direct-training mode selected.", flush=True)

command = [
    str(venv_python),
    "-u",
    str(PIPELINE_SCRIPT),
    "--config", str(CONFIG_PATH),
    "--source-root", str(RESEARCH_SOURCE_ROOT),
    "--train-script", str(TRAIN_DDP_SCRIPT),
    "--venv-python", str(venv_python),
]
print("+", " ".join(command), flush=True)
subprocess.run(command, check=True, env=environment)

## 5. Final academic results and downloadable ZIP

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

final_results_path = WORK_ROOT / 'FINAL_RESULTS.json'
zip_meta_path = WORK_ROOT / 'FINAL_ZIP.json'
academic_summary_path = WORK_ROOT / 'academic' / 'ACADEMIC_SUMMARY.md'

if not final_results_path.is_file():
    raise FileNotFoundError(f'Missing final result: {final_results_path}')
if not ZIP_PATH.is_file():
    raise FileNotFoundError(f'Missing final ZIP: {ZIP_PATH}')

final_results = json.loads(final_results_path.read_text(encoding='utf-8'))
zip_meta = json.loads(zip_meta_path.read_text(encoding='utf-8'))

display(Markdown(academic_summary_path.read_text(encoding='utf-8')))

overview = pd.read_csv(WORK_ROOT / 'academic' / 'tables' / 'experiment_overview_percent.csv')
matched = pd.read_csv(WORK_ROOT / 'academic' / 'tables' / 'same_shrimpdb_test_domain_comparison.csv')
reference_path = WORK_ROOT / 'academic' / 'tables' / 'combined_vs_reported_processed_only_reference.csv'

display(overview)
display(matched)
if reference_path.is_file():
    display(pd.read_csv(reference_path))

print('FINAL APPLICATION CHECKPOINT:')
print(final_results['final_application_checkpoint'])
print('CHECKPOINT SHA-256:')
print(final_results['final_application_checkpoint_sha256'])
print('\nFINAL ZIP:')
print(zip_meta['zip_path'])
print('ZIP SHA-256:')
print(zip_meta['zip_sha256'])
print('ZIP SIZE BYTES:')
print(zip_meta['zip_size_bytes'])


## Output structure

```text
/kaggle/working/cvio_shrimp_academic_study_seed42/
├── audit/                         # corruption, count, conflict, duplicate and leakage evidence
├── configs/                       # unified study config + effective experiment configs
├── experiments/                   # split manifests and source/class distributions
├── runs/                          # full Ultralytics outputs, plots, results.csv, best.pt, last.pt
├── evaluation/                    # predictions, metrics, CIs and confusion matrices
├── academic/                      # academic summary, percentage tables and comparison figures
├── source_snapshot/               # exact reviewed custom loss/attention implementation
├── final_application_model/       # combined four-class best.pt + class mapping
└── FINAL_RESULTS.json

/kaggle/working/CVio_ShrimpDB_Combined_ASL_LDAM_SimAM_DCFR_seed42_RESULTS.zip
```

The prepared image trees are intentionally excluded from the ZIP to avoid duplicating Kaggle input data. Checkpoints, default Ultralytics plots, results tables, source snapshot, manifests, audits and academic outputs are retained.